**PART 1.  FROM TEXT TO NUMBERS**

- OUR TINY CORPUS
  * d1 : "the movie was great"             label 1  (positive)
  * d2 : "the movie was terrible"          label 0  (negative)
  * d3 : "great great acting"              label 1
  * d4 : "the acting was terrible"         label 0


A model can only do arithmetic.  So every NLP system does the same chain:

     raw text -> clean -> TOKENS -> VOCABULARY (token -> id) -> numbers -> model

1.1  TEXT PREPROCESSING  (clean the text)
  - lowercase ("Great" and "great" become one word)
  - remove punctuation and extra spaces
  - (optional) remove STOP WORDS such as "the", "a", "is"
  - (optional) STEMMING / LEMMATISING: "running", "runs" -> "run"
  Caution: cleaning can destroy meaning.  "not good" without "not" says the
  opposite.  Modern LLMs do almost NO cleaning; they let the model learn it.

1.2  TOKENIZATION  (cut the text into pieces)
  - WORD tokens:       "the movie was great" -> [the, movie, was, great]
  - CHARACTER tokens:  "great" -> [g, r, e, a, t]       (long sequences)
  - SUBWORD tokens (what GPT-style models use):  "unhappiness" -> [un, happi, ness]
        rare words are split into pieces the model has seen before, so there is
        no "unknown word".  A token is NOT always a word; about 4 characters or
        three quarters of an English word on average.  You will meet this again
        on Days 20 and 21 (context window, cost per token).

1.3  VOCABULARY  (a table: token -> id)
  - Collect the unique tokens, sort them, give each one an integer id.
  - Two SPECIAL tokens are reserved:  <PAD> = 0  (filler, Part 5)  and
    <UNK> = 1  (a word not in the vocabulary).  Real words start at id 2.
  - The model never sees the word; it sees the id.

  EX 1 (paper, 10 min).  Tokenise d1 to d4 by splitting on spaces (they are
  already lowercase).  Build the vocabulary in ALPHABETICAL order with ids
  starting at 2:

       word      id          word      id          word      id
       acting    2           great     3           movie      4
       terrible  5           the       6           was        7

  Then write each document as a list of ids.
  
	 * d1: (6, 4, 7, 3)
	 * d2: (6, 4, 7, 5)
	 * d3: (3, 3, 2)
	 * d4: (6, 2, 7, 5)

**PART 2.  BAG OF WORDS**

A document becomes a COUNT VECTOR with one position per vocabulary word.
The word order is thrown away (hence "bag").  Use the alphabetical word order
of EX 1 for the positions:   [acting, great, movie, terrible, the, was]

d1 "the movie was great"  ->  [0, 1, 1, 0, 1, 1]

EX 2 (paper, 10 min).  Write the Bag-of-Words vector of d2, d3 and d4.

* d2 "the movie was terible"    -> [0, 0, 1, 1, 1, 1]
* d3 "great great acting"       -> [1, 2, 0, 0, 0, 0]
* d4 "the acting was terrible"  -> [1, 0, 0, 1, 1, 1]

COSINE SIMILARITY  (how alike two vectors are, from 0 to 1 for counts)

    cos(a, b) = (a . b) / ( |a| x |b| )          |a| = square root of the sum of squares

Worked: d1 = [0,1,1,0,1,1], d2 = [0,0,1,1,1,1].
    a . b = 0+0+1+0+1+1 = 3 ;  |d1| = sqrt(4) = 2 ;  |d2| = 2 ;  cos = 3/4 = 0.75

EX 3 (paper, 5 min).  Compute cos(d3, d4).  Then say which pair of documents is
the MOST similar and whether that makes sense for the labels.
*(d1, d2) and (d2, d4) are the most similar and makes sense since it is similar to 75%. 

* cos(d3, d4) = (1 * 1) + (2 * 0) + (0 * 1) + (0 * 1) + (0 * 1) + (0 * 1) = 1 + 0 + 0 + 0 + 0 + 0 = 1
            = 1 / (2.236 * 2) = 0.224
* cos(d2, d4) = 0.75
* cos(d1, d4) = 0.50
* cos(d1, d3) = 0.447
* cos(d2, d3) = 0.0

LIMITS OF BAG OF WORDS
- order is lost ("dog bites man" = "man bites dog"), negation is lost
- common words ("the", "was") count as much as informative ones
- the vectors are as long as the vocabulary (50,000 words -> 50,000 numbers,
almost all zeros) and "great" and "excellent" are unrelated columns

**PART 3.  TF-IDF**

Idea: a word is IMPORTANT for a document if it appears often in THIS document
(TF) but in FEW documents overall (IDF).

* TF(w, d)  = count of w in d / number of tokens in d
* IDF(w)    = ln( N / df(w) )        N = number of documents, df = documents containing w
* TF-IDF    = TF x IDF

(ln is the natural logarithm.)  This is the textbook form.  sklearn's
TfidfVectorizer uses a smoothed IDF, ln((1+N)/(1+df)) + 1, and then scales each
vector to length 1, so ITS numbers differ from ours; the idea is identical.

Worked: N = 4.  "great" appears in d1 and d3, so df = 2 and
      IDF(great) = ln(4/2) = 0.6931 ;  IDF(the) = ln(4/3) = 0.2877

EX 4 (paper, 15 min).

a) df for all 6 words, and their IDF.
   * acting: 2, IDF: len(4/2) -> 0.6931
   * the: 3, IDF: len(4/3) -> 0.2877
   * great: 2, IDF: len(4/2) -> 0.6931
   * movie: 2, IDF: len(4/2) -> 0.6931
   * terrible: 2, IDF: len(4/2) -> 0.6931
   * was: 3, IDF: len(4/3) -> 0.2877

b) TF-IDF vector of d1 (positions as in EX 2).  [TF is 1/4 for each word of d1.]
   * acting: 0 x 0.6931, the: 0.25 x 0.2877, great: 0.25 x 0.6931, movie: 0.25 x 0.6931
   terrible: 0 x 0.6931, was: 0.25 x 0.2877 -> [0.0, 0.1733, 0.1733, 0.0, 0.0719, 0.0719]

c) TF-IDF vector of d3.  [d3 has 3 tokens: TF(great) = 2/3, TF(acting) = 1/3.]
   * great: 0.6667 x 0.6931, acting: 0.3333 x 0.6931 -> [0.2310, 0.4621, 0.0, 0.0, 0.0, 0.0]

d) Which word has the SMALLEST weight in d1, and why?
   * the and was because all four words present in $d_1$ have the exact same Term Frequency (TF = 0.25). However, the and was appear in 3 out of 4 documents (df = 3), giving them a lower Inverse Document Frequency (IDF 0.2877) than great and movie (df = 2$, IDF approx 0.6931). 